# USP5 — Uji Label Diacak 100 Kali untuk Model Linear

**Dijalankan di kernel yang sama**, setelah notebook `USP5_Uji_Anggota_Ensemble` selesai (memakai `buat_anggota`, `buat_fold_uji`, `indeks_latih`, dan variabel lain dari sana).

## Untuk apa?

Ujian sebelumnya hanya 20 pengacakan, sehingga nilai p terkecil yang mungkin adalah 0,048. Dengan **100 pengacakan**, nilai p terkecilnya sekitar **0,0099**, jadi klaim "sinyalnya nyata" jauh lebih kuat.

Ujian ini dilakukan untuk dua anggota linear yang terbukti paling bagus (**BayesianRidge** dan **SVR linear**), dan untuk dua skenario (A: latih Mann saja, B: latih semua). Dua ukuran dicek: **R²** dan **Spearman** (ketepatan urutan, yang paling penting untuk memilih kandidat).

GradientBoosting sengaja tidak diikutkan karena lambat dan sudah terbukti lemah. Ensemble penuh sudah diuji di sel sebelumnya (20 pengacakan).

## 1. Cek prasyarat

**Apa ini?** Memastikan semua variabel dan fungsi dari notebook sebelumnya masih ada. Kalau ada yang hilang, sel ini berhenti dan menyebutkan apa yang harus dijalankan ulang.

In [ ]:
perlu = ['X', 'y', 'groups', 'idx53', 'buat_anggota', 'buat_fold_uji', 'indeks_latih',
         'SEEDS', 'SKENARIO', 'OUT', 'r2_score', 'spearmanr', 'np', 'pd', 'plt']
hilang = [n for n in perlu if n not in globals()]
assert not hilang, f'Jalankan dulu notebook sebelumnya. Yang hilang: {hilang}'
print('Prasyarat lengkap | senyawa uji:', len(idx53), '| skenario:', SKENARIO)

## 2. Fungsi skor untuk satu anggota

**Apa ini?** Fungsi yang melatih **satu anggota** (misalnya BayesianRidge) dengan aturan ujian yang **sama persis** seperti sebelumnya (senyawa uji hanya senyawa Mann, scaffold tidak bocor), lalu mengembalikan R² dan Spearman. Karena hanya model linear, prosesnya cepat.

In [ ]:
def skor_anggota(nama, skenario, kind, seed):
    pos = {int(i): p for p, i in enumerate(idx53)}
    pred = np.full(len(idx53), np.nan)
    for uji in buat_fold_uji(kind, seed):
        latih = indeks_latih(skenario, uji, kind)
        assert set(latih).isdisjoint(set(uji)), 'Data uji bocor ke data latih'
        if kind == 'scaffold':
            assert set(groups[latih]).isdisjoint(set(groups[uji])), 'Scaffold bocor'
        est = buat_anggota()[nama]
        est.fit(X[latih], y[latih])
        for i, v in zip(uji, np.ravel(est.predict(X[uji]))):
            pred[pos[int(i)]] = v
    yt = y[idx53]
    rho = spearmanr(yt, pred).correlation
    return float(r2_score(yt, pred)), float(np.nan_to_num(rho))

KANDIDAT = ['BayesianRidge', 'SVR_linear']
SEED_Y = SEEDS[0]
print('Anggota diuji:', KANDIDAT, '| seed pembagian data:', SEED_Y)

## 3. Skor asli (label benar)

**Apa ini?** Menghitung skor dengan **label pKd yang asli** untuk setiap anggota dan skenario. Nilai ini yang nanti dibandingkan dengan hasil label acak. Pembagian data (fold) sama persis dengan yang dipakai pada ujian acak, supaya perbandingannya adil.

In [ ]:
asli = {}
for sk in SKENARIO:
    for nama in KANDIDAT:
        asli[(nama, sk)] = skor_anggota(nama, sk, 'scaffold', SEED_Y)
        print(f'{nama:14s} | {sk:24s} | R2 asli = {asli[(nama, sk)][0]:.3f} | Spearman asli = {asli[(nama, sk)][1]:.3f}')

## 4. Mengocok label 100 kali

**Apa ini?** Inti ujiannya. Pada tiap pengulangan, **nilai pKd dikocok antar senyawa** (struktur tetap, jawabannya ditukar acak), lalu model dilatih dan diuji lagi dengan aturan yang sama. Kalau model benar-benar menangkap hubungan struktur dan aktivitas, skor pada label acak harus jatuh (R² sekitar nol atau negatif).

Pengacakan dilakukan **di tempat**, dan `y` asli **dikembalikan otomatis di akhir** (dan dicek), bahkan kalau terjadi error di tengah jalan. Waktu proses: sekitar satu sampai beberapa menit.

In [ ]:
N_PERM = 100
rng = np.random.RandomState(2026)
y_asli = y.copy()
acak = {(n, s): [] for n in KANDIDAT for s in SKENARIO}

try:
    for i in range(N_PERM):
        y[:] = y_asli[rng.permutation(len(y_asli))]          # kocok jawaban (in-place)
        for sk in SKENARIO:
            for nama in KANDIDAT:
                acak[(nama, sk)].append(skor_anggota(nama, sk, 'scaffold', SEED_Y))
        if (i + 1) % 20 == 0:
            print(f'pengacakan {i + 1}/{N_PERM} selesai')
finally:
    y[:] = y_asli                                             # kembalikan y asli, apa pun yang terjadi
assert np.allclose(y, y_asli), 'y asli gagal dikembalikan!'
print('y asli sudah dikembalikan dan dicek.')

## 5. Tabel hasil dan nilai p

**Apa ini?** Merangkum hasil. Cara membaca:

- **Skor asli harus jauh di atas** rata-rata dan maksimum skor acak.
- **Nilai p empiris** = (1 + jumlah pengacakan yang sama bagus atau lebih bagus dari yang asli) / (1 + 100). Nilai terkecil yang mungkin sekitar 0,0099. Makin kecil, makin kuat buktinya.
- Kolom `kesimpulan` otomatis menyebut apakah skor asli di atas semua hasil acak.

In [ ]:
baris = []
for (nama, sk), daftar in acak.items():
    arr = np.array(daftar)                    # kolom 0 = R2, kolom 1 = Spearman
    for k, metrik in enumerate(['R2', 'Spearman']):
        nilai_asli = asli[(nama, sk)][k]
        p_emp = (1 + (arr[:, k] >= nilai_asli).sum()) / (1 + len(arr))
        baris.append(dict(anggota=nama, skenario=sk, metrik=metrik, asli=round(nilai_asli, 3),
                          acak_rata2=round(arr[:, k].mean(), 3), acak_maks=round(arr[:, k].max(), 3),
                          acak_min=round(arr[:, k].min(), 3), p_empiris=round(p_emp, 4),
                          kesimpulan='di atas semua label acak' if nilai_asli > arr[:, k].max()
                                     else 'TIDAK jelas di atas label acak'))
tabel_p = pd.DataFrame(baris)
tabel_p.to_csv(OUT / 'y_randomization_100.csv', index=False)

nilai_acak = pd.DataFrame([dict(anggota=n, skenario=s, pengacakan=i + 1, R2=r, Spearman=rho)
                           for (n, s), d in acak.items() for i, (r, rho) in enumerate(d)])
nilai_acak.to_csv(OUT / 'y_randomization_100_nilai_acak.csv', index=False)

print(f'Nilai p terkecil yang mungkin dengan {N_PERM} pengacakan: {1 / (1 + N_PERM):.4f}\n')
display(tabel_p)

## 6. Grafik

**Apa ini?** Histogram skor R² dari 100 label acak (abu-abu) dan garis merah untuk skor asli. Satu panel untuk tiap kombinasi anggota dan skenario. Kalau garis merah jauh di kanan seluruh batang abu-abu, sinyalnya nyata.

In [ ]:
fig, ax = plt.subplots(len(SKENARIO), len(KANDIDAT), figsize=(11, 6), squeeze=False)
for r, sk in enumerate(SKENARIO):
    for c, nama in enumerate(KANDIDAT):
        a = ax[r][c]
        arr = np.array(acak[(nama, sk)])[:, 0]
        a.hist(arr, bins=20, color='#999999', edgecolor='white', label='label diacak (100x)')
        a.axvline(asli[(nama, sk)][0], color='#C44E52', lw=2.5, label=f'asli ({asli[(nama, sk)][0]:.3f})')
        a.axvline(0, color='k', lw=1, ls=':')
        a.set_title(f'{nama} | {sk.replace("_", " ")}', fontsize=9)
        a.set_xlabel('R2 out-of-fold'); a.set_ylabel('frekuensi')
        a.legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUT / 'grafik_y_randomization_100.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Cara melaporkan dengan jujur

1. **Yang dibuktikan ujian ini:** model menangkap hubungan struktur dan aktivitas yang nyata pada 53 senyawa ini, bukan korelasi kebetulan. Ini **tidak** membuktikan bahwa model akan akurat untuk seri senyawa lain (misalnya stilben).
2. **Kalimat yang aman** (sesuaikan dengan hasilmu): "Uji label diacak dengan 100 pengacakan menunjukkan bahwa R² dan Spearman model BayesianRidge dan SVR linear berada di atas seluruh hasil label acak (p empiris ≈ 0,01)."
3. **Batasannya:** satu seed pembagian data, senyawa uji yang sama dipakai berulang, dan pengacakan dilakukan pada semua 80 label. Tulis ini di bagian keterbatasan.
4. **Ingat aturan sebelumnya:** hasil utama tetap ensemble penuh. Temuan bahwa model linear sama baiknya atau lebih baik dilaporkan sebagai analisis eksplorasi.